# 01 Create a sample and translate

Translates the open answers into English with an LLM on Nebula and draws a stratified sample for manual annotation. The sample is written to an Excel sheet with one column per code of the codebook, which serves as the gold standard.

Cells tagged `api` call the LLM (OpenAI or Nebula) and were **not executed** for this repository. `dummy_data/` holds 12 synthetic citizen answers (plus 4 expert rows), random coding runs and random manual codings with the original file names. All results shown are meaningless.

In [1]:
# Folders. The original notebook read and wrote files in the project folder:
# DATA_DIR = OUT_DIR = "."
DATA_DIR = "dummy_data"   # synthetic example data shipped with this repo
OUT_DIR = "output"        # written by the notebook (gitignored)
import os; os.makedirs(OUT_DIR, exist_ok=True)

# ---- API switch -------------------------------------------------------------
# Cells starting with %%api call an external API (OpenAI, Nebula, Qualtrics, Eurostat, TypeSafe).
# RUN_API = False: these cells are skipped, so "Run All" makes no API calls and changes nothing on a server.
# RUN_API = True:  they run normally (needs the API keys; see README).
RUN_API = False
from IPython.core.magic import register_cell_magic
@register_cell_magic
def api(line, cell):
    if RUN_API:
        get_ipython().run_cell(cell)
    else:
        print("API cell skipped (RUN_API = False)")

In [2]:
import pandas as pd
import MaxNLP
from pathlib import Path
from tqdm.auto import tqdm
import json
tqdm.pandas(desc="Applying")
%load_ext autoreload
%autoreload 2
    
from datetime import date
date_str = date.today().isoformat()

In [3]:
import re
#df=pd.read_json(f"{DATA_DIR}/clean_df.json")
df=pd.read_json(f"{DATA_DIR}/clean_df.json")

# replace whitespace with single whitespace
cols=["QID8"] # causes
cols=["QID19"] # policy suggestions
df[cols] = df[cols].map(lambda x: re.sub(r'\s+', ' ', x).strip() if isinstance(x, str) else "")


In [4]:
df=pd.read_json(f"{DATA_DIR}/2026-06-23 all translated.json")

In [5]:
%%api
# Load the Nebula Client
from dotenv import load_dotenv
import openAI_key

client=MaxNLP.create_client(client="Nebula", openAI_key=openAI_key)
available_models = MaxNLP.get_nebula_models(client)

print(f"Models: {available_models}\n\n")


API cell skipped (RUN_API = False)


In [6]:
%%api
import pandas as pd
from tqdm.auto import tqdm

system_prompt = """
You are a professional translator.

Translate each citizen survey response to English.

Rules:
1. Preserve meaning, tone, wording and informal style.
2. Do not explain or interpret.
3. Keep numbering exactly the same.
4. Return one translation for every input.
5. Output ONLY lines formatted as

<ID>\t<translation>

Example:
0\tEnergy prices increased because...
1\tThe war caused...
"""

# Approximation: 1 token ≈ 4 characters
CHARS_PER_TOKEN = 4
MAX_INPUT_TOKENS = 3000   # Leave room for output

def estimate_tokens(text):
    return max(1, len(str(text)) // CHARS_PER_TOKEN)


def make_batches(df, text_col, max_tokens=MAX_INPUT_TOKENS):

    batches = []
    current = []
    current_tokens = 0

    for idx, text in zip(df.index, df[text_col]):

        t = estimate_tokens(text)

        if current and current_tokens + t > max_tokens:
            batches.append(current)
            current = []
            current_tokens = 0

        current.append((idx, text))
        current_tokens += t

    if current:
        batches.append(current)

    return batches


def translate_batch(batch,
                    model="FAST.gpt-oss:120b",
                    prompt=system_prompt):

    user_prompt = "\n".join(
        f"{idx}\t{text}"
        for idx, text in batch
    )

    response = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user",
             "content": f"Translate the following responses:\n\n{user_prompt}"}
        ]
    )

    output = response.choices[0].message.content.strip()

    translations = {}

    for line in output.splitlines():
        if "\t" not in line:
            continue
        idx, translation = line.split("\t", 1)
        translations[int(idx)] = translation.strip()

    return translations

batches = make_batches(df, "QID8")

print(f"{len(batches)} batches")
print(f"Average batch size: {sum(len(b) for b in batches)/len(batches):.1f}")

translations = {}

#for batch in tqdm(batches):
#    translations.update(translate_batch(batch))

#df.loc[df["QID8_en"].isna(),"QID8_en"] = df.loc[df["QID8_en"].isna(),"QID8"].index.map(translations)

API cell skipped (RUN_API = False)


In [7]:
MAX_INPUT_TOKENS = 1000

batches = make_batches(df, "QID19",max_tokens = MAX_INPUT_TOKENS)
print(f"{len(batches)} batches")
print(f"Average batch size: {sum(len(b) for b in batches)/len(batches):.1f}")


NameError: name 'make_batches' is not defined

In [8]:
import pickle 

with open(f'{OUT_DIR}/Futures_translation.pkl', 'wb') as f:
    pickle.dump(translations, f)
        
with open(f'{DATA_DIR}/Futures_translation.pkl', 'rb') as f:
    loaded_dict = pickle.load(f)


NameError: name 'translations' is not defined

In [9]:
loaded_dict

NameError: name 'loaded_dict' is not defined

In [10]:
index=df.index #translations
for i in index:
    if i in translations.keys():
        df["QID19_en"]=translations[i]
    else:
        print(i)

NameError: name 'translations' is not defined

In [11]:
df.QID19_en

AttributeError: 'DataFrame' object has no attribute 'QID19_en'

In [12]:
translations = {}

for batch in tqdm(batches):
    translations.update(translate_batch(batch))

NameError: name 'batches' is not defined

In [13]:
df.loc[df["QID19_en"].isna(),"QID19_en"]

KeyError: 'QID19_en'

In [14]:
df.loc[df["QID19_en"].isna(),"QID19_en"] = df.loc[df["QID19_en"].isna(),"QID19"].index.map(translations)

df.to_json(f"{OUT_DIR}/2026-07-19 futures translated.json")

KeyError: 'QID19_en'

In [15]:
%%api

def translate_batch(batch,
                    model="FAST.gpt-oss:120b",
                    prompt=system_prompt):

    user_prompt = "\n".join(
        f"{idx}\t{text}"
        for idx, text in zip(batch.index, batch.values)
    )
    
    print(user_prompt)
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user",
             "content": f"Translate the following responses:\n\n{user_prompt}"}
        ]
    )

    output = response.choices[0].message.content.strip()

    translations = {}

    for line in output.splitlines():
        if "\t" not in line:
            continue
        idx, translation = line.split("\t", 1)
        translations[int(idx)] = translation.strip()

    return translations


batch=df.loc[df["QID8_en"].isna(),"QID8"]

translation=translate_batch(batch)

df.loc[df["QID8_en"].isna(),"QID8_en"]=pd.Series(translation)

API cell skipped (RUN_API = False)


In [16]:
df

,StartDate,EndDate,Status,Progress,Duration (in seconds),Finished,RecordedDate,ResponseId,DistributionChannel,UserLanguage,...,income_num,expertize_num,politics_num,house_density_num,house_energy_num,urban_num,User-ID,User-Country,expert,QID8_en
0,Start Date,End Date,Response Type,Progress,Duration (in seconds),Finished,Recorded Date,Response ID,Distribution Channel,User Language,...,household income,expertise in energy topics,politically right,What type of residence do you live in?,How would you describe your typical energy usa...,urbanity,Q_URL,Q_URL,False,What do you think caused the European energy c...
1,"{""ImportId"":""startDate"",""timeZone"":""America/De...","{""ImportId"":""endDate"",""timeZone"":""America/Denv...","{""ImportId"":""status""}","{""ImportId"":""progress""}","{""ImportId"":""duration""}","{""ImportId"":""finished""}","{""ImportId"":""recordedDate"",""timeZone"":""America...","{""ImportId"":""_recordId""}","{""ImportId"":""distributionChannel""}","{""ImportId"":""userLanguage""}",...,"{""ImportId"":""QID26""}","{""ImportId"":""QID31""}","{""ImportId"":""QID33""}","{""ImportId"":""QID29""}","{""ImportId"":""QID30""}","{""ImportId"":""QID28""}","{""ImportId"":""Q_URL""}","{""ImportId"":""Q_URL""}",False,NaN
2,2000-01-04 00:13:00,2000-01-04 00:22:29,IP Address,100,569,True,2000-01-04 00:22:29,R_EXAMPLE110000,anonymous,FR,...,2.0,1.0,None,1.0,None,4.0,example-11-0000,EN,False,Gas prices rose because supply from Russia sto...
3,2000-01-19 03:49:00,2000-01-19 03:54:35,IP Address,100,335,True,2000-01-19 03:54:35,R_EXAMPLE110001,anonymous,DE,...,4.0,3.0,2.0,2.0,2.0,4.0,example-11-0001,EN,False,Gas prices rose because supply from Russia sto...
4,2000-01-08 03:31:00,2000-01-08 04:06:45,IP Address,100,2145,True,2000-01-08 04:06:45,R_EXAMPLE110002,anonymous,IT,...,4.0,3.0,1.0,3.0,1.0,3.0,example-11-0002,EN,False,Die Strompreise sind stark gestiegen.
5,2000-01-19 14:03:00,2000-01-19 14:20:06,IP Address,100,1026,True,2000-01-19 14:20:06,R_EXAMPLE110003,anonymous,NL,...,1.0,3.0,3.0,2.0,3.0,3.0,example-11-0003,EN,False,More solar and wind would help.
6,2000-01-20 09:27:00,2000-01-20 09:51:48,IP Address,100,1488,True,2000-01-20 09:51:48,R_EXAMPLE110004,anonymous,CS,...,5.0,1.0,None,1.0,1.0,1.0,example-11-0004,EN,False,I don't know.
7,2000-01-06 06:29:00,2000-01-06 06:41:40,IP Address,100,760,True,2000-01-06 06:41:40,R_EXAMPLE110005,anonymous,ES-ES,...,2.0,1.0,1.0,4.0,1.0,1.0,example-11-0005,EN,False,I don't know.
8,2000-01-10 07:31:00,2000-01-10 07:52:43,IP Address,100,1303,True,2000-01-10 07:52:43,R_EXAMPLE110006,anonymous,PL,...,4.0,5.0,1.0,1.0,3.0,1.0,example-11-0006,EN,False,The EU should buy energy together.
9,2000-01-04 00:17:00,2000-01-04 00:53:07,IP Address,100,2167,True,2000-01-04 00:53:07,R_EXAMPLE110007,anonymous,PL,...,1.0,2.0,1.0,1.0,2.0,4.0,example-11-0007,EN,False,Energy bills went up a lot after 2022.


In [17]:
df.to_json(f"{OUT_DIR}/2026-06-23 all translated.json")

# create sample sheet for manual Annotation

In [18]:
codebook_causes = Path("codebooks/2026-05-06 causes codebook.md").read_text(encoding="utf-8")
codebook_causes_json=MaxNLP.markdown_to_json(codebook_causes)
codes_causes= [i["label"] for i in codebook_causes_json]
codes_causes

['EU political actors',
 'National political actors',
 'Industry and market actors',
 'Media actors',
 "Russia's war on Ukraine",
 'Too-fast energy transition',
 'Dependence on fossil fuels',
 'Concentrated energy import dependency',
 'Uncontrollable natural disasters',
 'External shocks and technical failures',
 'Abstract societal and ideological explanations',
 'EU Sanctions against Russia',
 'nuclear phase-out / reduced national production',
 'Insufficient price regulation']

In [19]:
import pandas as pd

n = len(df_t)
cols =["labels"]+ df_t.index.tolist()

codes=codes_causes
annotation_df = pd.DataFrame(
    [[c] + [0]*n for c in codes] +
    [[""] + [""]*n] +
    [["text"] + df_t["QID8"].tolist()] +
    [["translation"] + df_t["QID8_en"].tolist()],
    columns=cols
)

annotation_df.to_excel(f"{OUT_DIR}/2026-05-14 causes_manual_annotation.xlsx", index=False, freeze_panes=(1, 1))

NameError: name 'df_t' is not defined

In [20]:
#QID8 == Causes
